In [6]:
import torch
import os

from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision import models
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
import matplotlib.pyplot as plt



#Importieren der Daten
def create_sample():
    sample = []
    jubaea_directory = "Data/Jubaea"
    for image in os.listdir(jubaea_directory):
        if image.lower().endswith((".jpg", ".jpeg", ".png")):
            sample.append((os.path.join(jubaea_directory, image),1))


    not_jubaea_directory = "Data/Not_Jubaea"
    for subfolder in os.listdir(not_jubaea_directory):
        subfolder_path = os.path.join(not_jubaea_directory, subfolder)
        if not os.path.isdir(subfolder_path):
            continue
        for image in os.listdir(os.path.join(not_jubaea_directory, subfolder)):
            if image.lower().endswith((".jpg", ".jpeg", ".png")):
                sample.append((os.path.join(not_jubaea_directory,subfolder,image),0))
    return sample

sample = create_sample()
pictures = [x[0] for x in sample]
label = [x[1] for x in sample]
X_train, X_test, Y_train, Y_test = train_test_split(pictures, label, test_size=0.2, stratify=label, random_state=42)

#Data Augmentation im Training durch RandomRotation
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomRotation(15),
    transforms.RandomHorizontalFlip(0.5),
    transforms.ToTensor(),
])

#Resize von den Bildern zu 224x224 für resnet18
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])
#Klasse Datasets
class JubaeaDataset(Dataset):
    def __init__(self, path, labels, transform):
        self.path = path
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.path)

    def __getitem__(self, index):
        path = self.path[index]
        labels = self.labels[index]

        image = Image.open(path).convert("RGB")
        labels = torch.tensor(labels, dtype=torch.long)

        if self.transform:
            image = self.transform(image)

        return image, labels

train_dataset = JubaeaDataset(X_train, Y_train, transform=train_transform)
test_dataset = JubaeaDataset(X_test, Y_test, transform=test_transform)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)


#Neuronales Netzwerk von resnet18 benutzen
net = models.resnet18(models.ResNet18_Weights.DEFAULT)
net.load_state_dict(torch.load("Jubaea_resnet.pth"))

#Test  mit gespeichertem Model
net.eval()
pre_correct = 0
pre_total = 0

with torch.no_grad():
    for images, labels in test_loader:
        outputs = net(images)
        preds = torch.argmax(outputs, dim=1)
        pre_correct += (preds == labels).float().sum()
        pre_total += labels.size(0)

model_acc = (pre_correct / pre_total).item()
print("Acc vom Model beträgt", model_acc)

#Lossfunktion
loss_fn = nn.CrossEntropyLoss()

#Optimierung der Gewichte
optimizer = torch.optim.Adam(net.parameters(), lr=0.0001, weight_decay=5e-4)


#Trainingsschleife mit loss und Accuracy
epochs = 5
best_acc = model_acc

#Klassifizierung, Jubaea oder nicht Jubaea
classes = ["Not_Jubaea", "Jubaea"]

for epoch in range (epochs):
    loss_calc = 0.0
    correct_train = 0
    total_train = 0
    correct = 0
    total_test = 0
    net.train()
    for images, labels in train_loader:
        optimizer.zero_grad()
        outputs = net(images)

        loss = loss_fn(outputs, labels)
        loss.backward()
        optimizer.step()
        loss_calc += loss.item()

        predict_train = torch.max(outputs, 1)[1]
        correct_train += (predict_train == labels).sum().item()
        total_train += labels.size(0)

    epoch_loss = loss_calc / len(train_loader)

    train_acc = correct_train / total_train

    print(f"Durchlauf {epoch+1} fertig.")
    print("Der Trainingsloss beträgt", epoch_loss)
    print("Die Trainingsgenauigkeit beträgt: ",train_acc)


#Testschleife mit Model speichern und Fehlklassifikationen zeigen
    net.eval()
    with torch.no_grad():
        for images, labels in test_loader:
            outputs = net(images)
            probs =  torch.softmax(outputs, dim=1)
            preds = torch.argmax(outputs, dim=1)

            predict = torch.max(outputs, 1)[1]
            correct += (predict == labels).sum().item()
            total_test += labels.size(0)

            # Zeigen von Fehlklassifikationen, bei mehreren Epochen ausschalten
            """"
            for i in range(len(images)):
                if preds[i] != labels[i]:
                    plt.imshow(images[i].permute(1, 2, 0))
                    plt.title(
                        f"falsch\n"
                        f"Zugehörige Klasse: {classes[labels[i].item()]}\n"
                        f"Vorhersage: {classes[preds[i].item()]} | "
                        f"P(Jubaea)={probs[i][1].item():.3f}"
                    )
                    plt.axis('off')
                    plt.show()
"""

        acc = correct / total_test
        if acc > best_acc:
            best_acc = acc
            torch.save(net.state_dict(), "Jubaea_resnet.pth")
            print("Ein Modell wurde gespeichert.")

        print("Die Testgenauigkeit beträgt: ", acc,".")


/home/cristian/.local/lib/python3.13/site-packages/torchvision/models/_utils.py:135: UserWarning: Using 'weights' as positional parameter(s) is deprecated since 0.13 and may be removed in the future. Please use keyword parameter(s) instead.
  warnings.warn(


Acc vom Model beträgt 0.7476635575294495
Durchlauf 1 fertig.
Der Trainingsloss beträgt 0.6694932069097247
Die Trainingsgenauigkeit beträgt:  0.8782201405152225
Ein Modell wurde gespeichert.
Die Testgenauigkeit beträgt:  0.794392523364486 .
Durchlauf 2 fertig.
Der Trainingsloss beträgt 0.17370485195091792
Die Trainingsgenauigkeit beträgt:  0.9601873536299765
Ein Modell wurde gespeichert.
Die Testgenauigkeit beträgt:  0.8598130841121495 .
Durchlauf 3 fertig.
Der Trainingsloss beträgt 0.15055061451026372
Die Trainingsgenauigkeit beträgt:  0.9648711943793911
Die Testgenauigkeit beträgt:  0.8598130841121495 .
Durchlauf 4 fertig.
Der Trainingsloss beträgt 0.05536543844001634
Die Trainingsgenauigkeit beträgt:  0.990632318501171
Die Testgenauigkeit beträgt:  0.8598130841121495 .
Durchlauf 5 fertig.
Der Trainingsloss beträgt 0.02699457760900259
Die Trainingsgenauigkeit beträgt:  0.9929742388758782
Ein Modell wurde gespeichert.
Die Testgenauigkeit beträgt:  0.8878504672897196 .
